# STJEPA V2 Ablation

- Creators: Sebastian Birk (sb75@sanger.ac.uk)
- Date of Creation: 30.06.2025
- Date of Last Modification: 07.04.2026 (by Sebastian Birk (sb75@sanger.ac.uk)

- Run this notebook in the stjepa_v2 uv environment, installable from ```../../envs/stjepa_v2.sh``` (on v2 branch).

## 1. Setup

### 1.1 Load Libraries 

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("../../utils")

In [3]:
import gc
import logging
import os
import warnings

import anndata as ad
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scib_metrics
import scipy.sparse as sp
import sklearn.metrics as skm
import torch
import squidpy as sq
from datasets import concatenate_datasets, load_from_disk
from itertools import combinations
from matplotlib.colors import Normalize
from scib_metrics import ilisi_knn
from scib_metrics.nearest_neighbors import pynndescent
from scipy.spatial.distance import cdist
from sklearn.metrics import silhouette_score
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score
from sklearn.preprocessing import StandardScaler

from app.inference import (embed_dataset,
                           harmonize_adata,
                           perturb_dataset,
                           tokenize_adata,
                           harmonize_tokenize_embed_pipeline,
                           get_gene_embed,
                           get_average_gene_embed,
                           get_spatial_score,
                           get_emd_distance)
from app.utils import init_model, load_checkpoint
from stjepa.utils.evaluation import (get_top_gene_score,
                                     get_top_gene_pairs)

from analysis_utils import *

/nfs/team361/sb75/.venvs/stjepa_v1/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/nfs/team361/sb75/.venvs/stjepa_v1/lib/python3.10/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/nfs/team361/sb75/.venvs/stjepa_v1/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


### 1.2 Define Config

In [285]:
# Ablation params
ablation_params = {}

# === TOKENIZER ===

#ablation_params['method'] = 'stjepa_v2_cell_graph_tokenizer'
#ablation_params['model_timestamp'] = '21032026_085513_196'
#ablation_params['latent_leiden_resolution'] = 1.32

#ablation_params['method'] = 'stjepa_v2_cell_neigh_tokenizer'
#ablation_params['model_timestamp'] = '09042026_164629_607'
#ablation_params['latent_leiden_resolution'] = 0.445

#ablation_params['method'] = 'stjepa_v2_cell_neigh_tokenizer_rank'
#ablation_params['model_timestamp'] = '10042026_193043_222'
#ablation_params['latent_leiden_resolution'] = 0.6

#ablation_params['method'] = 'stjepa_v2_cell_neigh_tokenizer_read_depth_shifted_log'
#ablation_params['model_timestamp'] = '11042026_024547_384'
#ablation_params['latent_leiden_resolution'] = 0.9

# === EXPRESSION ENCODING ===

#ablation_params['method'] = 'stjepa_v2_rank'
#ablation_params['model_timestamp'] = '09042026_115045_337'
#ablation_params['latent_leiden_resolution'] = 0.5

#ablation_params['method'] = 'stjepa_v2_rank_replica'
#ablation_params['model_timestamp'] = '08042026_083724_291'
#ablation_params['latent_leiden_resolution'] = 0.8

#ablation_params['method'] = 'stjepa_v2_counts_mlp'
#ablation_params['model_timestamp'] = '08042026_100918_294'
#ablation_params['latent_leiden_resolution'] = 0.7

#ablation_params['method'] = 'stjepa_v2_combined_mlp'
#ablation_params['model_timestamp'] = '21032026_085513_196'
#ablation_params['latent_leiden_resolution'] = 1.32

#ablation_params['method'] = 'stjepa_v2_combined_value_bins_20'
#ablation_params['model_timestamp'] = '09042026_182252_289'
#ablation_params['latent_leiden_resolution'] = 1.023

#ablation_params['method'] = 'stjepa_v2_combined_value_bins_50'
#ablation_params['model_timestamp'] = '10042026_193035_854'
#ablation_params['latent_leiden_resolution'] = 1.43

#ablation_params['method'] = 'stjepa_v2_combined_value_bins_100'
#ablation_params['model_timestamp'] = '11042026_013716_476'
#ablation_params['latent_leiden_resolution'] = 0.8

# === NORMALIZATION ===

#ablation_params['method'] = 'stjepa_v2_no_norm'
#ablation_params['model_timestamp'] = '24032026_224828_432'
#ablation_params['latent_leiden_resolution'] = 1.15

#ablation_params['method'] = 'stjepa_v2_shifted_log'
#ablation_params['model_timestamp'] = '24032026_224828_432'
#ablation_params['latent_leiden_resolution'] = 1.15

#ablation_params['method'] = 'stjepa_v2_read_depth_shifted_log'
#ablation_params['model_timestamp'] = '20032026_215239_257'
#ablation_params['latent_leiden_resolution'] = 1.32

#ablation_params['method'] = 'stjepa_v2_gene_corrected_read_depth_shifted_log'
#ablation_params['model_timestamp'] = '21032026_213950_914'
#ablation_params['latent_leiden_resolution'] = 1.8

#ablation_params['method'] = 'stjepa_v2_mean_no_norm_rank'
#ablation_params['model_timestamp'] = '23032026_204349_268'
#ablation_params['latent_leiden_resolution'] = 1.25

#ablation_params['method'] = 'stjepa_v2_seurat_no_norm_rank'
#ablation_params['model_timestamp'] = '22032026_135514_292'
#ablation_params['latent_leiden_resolution'] = 1.5

#ablation_params['method'] = 'stjepa_v2_mean'
#ablation_params['model_timestamp'] = '24032026_022512_769'
#ablation_params['latent_leiden_resolution'] = 1.725

#ablation_params['method'] = 'stjepa_v2_nz_mean'
#ablation_params['model_timestamp'] = '24032026_040014_694'
#ablation_params['latent_leiden_resolution'] = 1.5

#ablation_params['method'] = 'stjepa_v2_gene_corrected_read_depth_mean'
#ablation_params['model_timestamp'] = '21032026_190736_759'
#ablation_params['latent_leiden_resolution'] = 1.7

#ablation_params['method'] = 'stjepa_v2_gene_corrected_read_depth_nz_mean'
#ablation_params['model_timestamp'] = '21032026_182612_890'
#ablation_params['latent_leiden_resolution'] = 1.72

#ablation_params['method'] = 'stjepa_v2_seurat'
#ablation_params['model_timestamp'] = '21032026_154941_529'
#ablation_params['latent_leiden_resolution'] = 1.9

#ablation_params['method'] = 'stjepa_v2_pearson_residuals'
#ablation_params['model_timestamp'] = '21032026_182612_890'
#ablation_params['latent_leiden_resolution'] = 1.72

#ablation_params['method'] = 'stjepa_v2_gene_corrected_read_depth_mean_rank'
#ablation_params['model_timestamp'] = '23032026_233225_877'
#ablation_params['latent_leiden_resolution'] = 1.46

#ablation_params['method'] = 'stjepa_v2_mean_rank'
#ablation_params['model_timestamp'] = '23032026_220800_207'
#ablation_params['latent_leiden_resolution'] = 1.65

#ablation_params['method'] = 'stjepa_v2_nz_mean_rank'
#ablation_params['model_timestamp'] = '23032026_192023_653'
#ablation_params['latent_leiden_resolution'] = 1.6

#ablation_params['method'] = 'stjepa_v2_gene_corrected_read_depth_nz_mean_rank'
#ablation_params['model_timestamp'] = '24032026_005846_090'
#ablation_params['latent_leiden_resolution'] = 1.85

#ablation_params['method'] = 'stjepa_v2_nz_mean_rank'
#ablation_params['model_timestamp'] = '23032026_192023_653'
#ablation_params['latent_leiden_resolution'] = 0.7

#ablation_params['method'] = 'stjepa_v2_pearson_residuals_no_norm_rank'
#ablation_params['model_timestamp'] = '25032026_071947_719'
#ablation_params['latent_leiden_resolution'] = 1.2

#ablation_params['method'] = 'stjepa_v2_nz_mean_no_norm_rank'
#ablation_params['model_timestamp'] = '25032026_010748_327'
#ablation_params['latent_leiden_resolution'] = 1.75

#ablation_params['method'] = 'stjepa_v2_gene_corrected_read_depth_nz_mean_no_norm_rank'
#ablation_params['model_timestamp'] = '25032026_052014_360'
#ablation_params['latent_leiden_resolution'] = 1.75

#ablation_params['method'] = 'stjepa_v2_shifted_log_gene_corrected_read_depth_nz_mean_rank'
#ablation_params['model_timestamp'] = '24032026_204309_593'
#ablation_params['latent_leiden_resolution'] = 1.5

#ablation_params['method'] = 'stjepa_v2_pearson_residuals'
#ablation_params['model_timestamp'] = '21032026_172808_828'
#ablation_params['latent_leiden_resolution'] = 1.5

#ablation_params['method'] = 'stjepa_v2_no_norm_seurat_rank'
#ablation_params['model_timestamp'] = '08042026_193915_019'
#ablation_params['latent_leiden_resolution'] = 0.4005

#ablation_params['method'] = 'stjepa_v2_gene_corrected_read_depth_mean_no_norm_rank'
#ablation_params['model_timestamp'] = '25032026_031148_546'
#ablation_params['latent_leiden_resolution'] = 1.39

#ablation_params['method'] = 'stjepa_v2_no_norm_mean_rank'
#ablation_params['model_timestamp'] = '23032026_220800_207'
#ablation_params['latent_leiden_resolution'] = 1.5

#ablation_params['method'] = 'stjepa_v2_no_norm_nz_mean_rank'
#ablation_params['model_timestamp'] = '23032026_192023_653'
#ablation_params['latent_leiden_resolution'] = 1.5

#ablation_params['method'] = 'stjepa_v2_no_norm_pearson_residuals_rank'
#ablation_params['model_timestamp'] = '08042026_201620_913'
#ablation_params['latent_leiden_resolution'] = 1.25

#ablation_params['method'] = 'stjepa_v2_no_norm_gene_corrected_read_depth_nz_mean_rank'
#ablation_params['model_timestamp'] = '24032026_005846_090'
#ablation_params['latent_leiden_resolution'] = 1.85

#ablation_params['method'] = 'stjepa_v2_no_norm_gene_corrected_read_depth_mean_rank'
#ablation_params['model_timestamp'] = '23032026_233225_877'
#ablation_params['latent_leiden_resolution'] = 1.4530

# === SPATIAL ENCODING ===

#ablation_params['method'] = 'stjepa_v2_segment'
#ablation_params['model_timestamp'] = '21032026_085513_196'
#ablation_params['latent_leiden_resolution'] = 1.32

#ablation_params['method'] = 'stjepa_v2_coord_aligned'
#ablation_params['model_timestamp'] = '10042026_193042_980'
#ablation_params['latent_leiden_resolution'] = 2.0

#ablation_params['method'] = 'stjepa_v2_coord_aligned_09995ema'
#ablation_params['model_timestamp'] = '11042026_014620_031'
#ablation_params['latent_leiden_resolution'] = 2.0

# === SEQUENCE LENGTH ===

#ablation_params['method'] = 'stjepa_v2_rank_32_seq_len'
#ablation_params['model_timestamp'] = '08042026_230334_515'
#ablation_params['latent_leiden_resolution'] = 0.42

ablation_params['method'] = 'stjepa_v2_rank_64_seq_len'
ablation_params['model_timestamp'] = '08042026_083724_291'
ablation_params['latent_leiden_resolution'] = 0.8

#ablation_params['method'] = 'stjepa_v2_rank_128_seq_len'
#ablation_params['model_timestamp'] = '09042026_022225_041'
#ablation_params['latent_leiden_resolution'] = 1.4530

#ablation_params['method'] = 'stjepa_v2_rank_256_seq_len'
#ablation_params['model_timestamp'] = '08042026_132634_412'
#ablation_params['latent_leiden_resolution'] = 1.2

#ablation_params['method'] = 'stjepa_v2_rank_512_seq_len'
#ablation_params['model_timestamp'] = '08042026_232435_439'
#ablation_params['latent_leiden_resolution'] = 1.4

#ablation_params['method'] = 'stjepa_v2_combined_32_seq_len'
#ablation_params['model_timestamp'] = '09042026_033903_490'
#ablation_params['latent_leiden_resolution'] = 0.4

#ablation_params['method'] = 'stjepa_v2_combined_64_seq_len'
#ablation_params['model_timestamp'] = '21032026_085513_196'
#ablation_params['latent_leiden_resolution'] = 1.32

#ablation_params['method'] = 'stjepa_v2_combined_128_seq_len'
#ablation_params['model_timestamp'] = '09042026_023348_601'
#ablation_params['latent_leiden_resolution'] = 1.44

#ablation_params['method'] = 'stjepa_v2_combined_256_seq_len'
#ablation_params['model_timestamp'] = '11042026_013748_814'
#ablation_params['latent_leiden_resolution'] = 1.7

#ablation_params['method'] = 'stjepa_v2_combined_512_seq_len'
#ablation_params['model_timestamp'] = '09042026_030229_805'
#ablation_params['latent_leiden_resolution'] = 1.45

# === MASKING RATIO ===

#ablation_params['method'] = 'stjepa_v2_01_mask'
#ablation_params['model_timestamp'] = '11042026_073948_977'
#ablation_params['latent_leiden_resolution'] = 1.3

#ablation_params['method'] = 'stjepa_v2_02_mask'
#ablation_params['model_timestamp'] = '11042026_000356_147'
#ablation_params['latent_leiden_resolution'] = 0.44

#ablation_params['method'] = 'stjepa_v2_03_mask'
#ablation_params['model_timestamp'] = '10042026_193040_939'
#ablation_params['latent_leiden_resolution'] = 0.65

#ablation_params['method'] = 'stjepa_v2_04_mask'
#ablation_params['model_timestamp'] = '10042026_193005_683'
#ablation_params['latent_leiden_resolution'] = 1.17

#ablation_params['method'] = 'stjepa_v2_05_mask'
#ablation_params['model_timestamp'] = '09042026_143419_031'
#ablation_params['latent_leiden_resolution'] = 0.51

#ablation_params['method'] = 'stjepa_v2_06_mask'
#ablation_params['model_timestamp'] = '21032026_085513_196'
#ablation_params['latent_leiden_resolution'] = 1.32

#ablation_params['method'] = 'stjepa_v2_07_mask'
#ablation_params['model_timestamp'] = '09042026_115308_738'
#ablation_params['latent_leiden_resolution'] = 1.8

#ablation_params['method'] = 'stjepa_v2_07_mask_replica'
#ablation_params['model_timestamp'] = '11042026_010949_679'
#ablation_params['latent_leiden_resolution'] = 1.5

#ablation_params['method'] = 'stjepa_v2_08_mask'
#ablation_params['model_timestamp'] = '09042026_120745_754'
#ablation_params['latent_leiden_resolution'] = 1.6

#ablation_params['method'] = 'stjepa_v2_08_mask_replica'
#ablation_params['model_timestamp'] = '11042026_013603_079'
#ablation_params['latent_leiden_resolution'] = 1.2

#ablation_params['method'] = 'stjepa_v2_09_mask'
#ablation_params['model_timestamp'] = '09042026_134026_577'
#ablation_params['latent_leiden_resolution'] = 0.44

#ablation_params['method'] = 'stjepa_v2_09_mask_replica'
#ablation_params['model_timestamp'] = '11042026_000408_115'
#ablation_params['latent_leiden_resolution'] = 0.415

#ablation_params['method'] = 'stjepa_v2_095_mask'
#ablation_params['model_timestamp'] = '11042026_000356_187'
#ablation_params['latent_leiden_resolution'] = 0.35

#ablation_params['method'] = 'stjepa_v2_099_mask'
#ablation_params['model_timestamp'] = '11042026_000407_936'
#ablation_params['latent_leiden_resolution'] = 0.8

# === OTHER ===

#ablation_params['method'] = 'stjepa_test'
#ablation_params['model_timestamp'] = '05032026_155504_485'
#ablation_params['latent_leiden_resolution'] = 1.023

#ablation_params['method'] = 'stjepa_v2_no_norm_rank_tokenizer_seurat_tie_breaking'
#ablation_params['model_timestamp'] = '08042026_214009_816'
#ablation_params['latent_leiden_resolution'] = 1.3

#ablation_params['method'] = 'stjepa_v2_combined_4_enc_layers'
#ablation_params['model_timestamp'] = '11042026_030931_080'
#ablation_params['latent_leiden_resolution'] = 0.8


# General params
dataset_name = 'mmb0-1b_smb1-1b_1p'
batches = ['harmonised_merfish_mouse_brain_239-batch_batch82_shared_genes',
           'harmonised_starmap_plus_mouse_cns_batch15_shared_genes']
niche_key = 'tissue_region'
label_keys = [niche_key]
species = 'mouse'
n_neighs = 10
spot_sizes = {'MERFISH': 20,
              'STARmap PLUS': 100}
batch_key = 'batch'
n_pcs = None

# STJEPA params
emb_key = 'neighborhood_emb'
sample_key = None
min_cells_per_gene = 0
min_genes_per_cell = 10
nproc = 4
processing_mode = 'parallel'
use_generator = True
num_shards = 32
ensembl_release = 110
gene_mapping_dict_file_path = None
gene_occurrence_count_file_path = None
gene_occurrence_count_filter_value = 10
emb_layer = None
batch_size = 128
pin_memory = False
num_workers = 12
agg_excluded_genes = None
top_k = None
batch_key_new = 'batch_new'

### 1.3 Configure Paths

In [286]:
st_data_folder_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS'
st_data_silver_folder_path = f'{st_data_folder_path}/silver'
artifacts_folder_path = '/nfs/team361/sb75/stjepa-reproducibility/artifacts'
ablation_artifacts_folder_path = f'{artifacts_folder_path}/ablation'

cache_directory_path = f"{ablation_artifacts_folder_path}/{ablation_params['method']}/cache"
model_folder_path = f"{artifacts_folder_path}/models/{ablation_params['model_timestamp']}"

### 1.4 Run Notebook Setup

In [287]:
warnings.filterwarnings("ignore", category=ResourceWarning)

In [288]:
plt.rcParams['font.size'] = 5
plt.rcParams['text.usetex'] = False
plt.rcParams['svg.fonttype'] = 'none'

In [289]:
# Improve reproducibility of UMAP and Leiden
os.environ['NUMBA_CPU_NAME'] = 'generic'

pd.options.future.infer_string = False

In [290]:
sc.settings.figdir = f'{ablation_artifacts_folder_path}/figures'

sc.settings.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(3, 2),
    facecolor='white',
    fontsize=7
)

In [291]:
os.makedirs(f'{ablation_artifacts_folder_path}/figures', exist_ok=True)
os.makedirs(f'{ablation_artifacts_folder_path}/results', exist_ok=True)
os.makedirs(f"{ablation_artifacts_folder_path}/{ablation_params['method']}", exist_ok=True)
os.makedirs(f"{ablation_artifacts_folder_path}/{ablation_params['method']}/{ablation_params['model_timestamp']}", exist_ok=True)
os.makedirs(f"{ablation_artifacts_folder_path}/{ablation_params['method']}/cache", exist_ok=True)

### 1.5 Define Functions

In [292]:
def make_large_cmap(n):
    base = plt.cm.get_cmap("turbo", n)   # or "hsv", "nipy_spectral"
    return [mcolors.to_hex(base(i)) for i in range(n)]

In [293]:
def mmd_comparable(adata, emb_key, batch_key, n_sub=2000, n_sigma=1000):
    # Normalize to zero mean, unit variance
    X = StandardScaler().fit_transform(adata.obsm[emb_key])
    
    # Compute sigma on normalized data
    idx = np.random.choice(len(X), min(n_sigma, len(X)), replace=False)
    sub = X[idx]
    dists = np.linalg.norm(sub[:, None] - sub[None, :], axis=2)
    sigma = np.median(dists[dists > 0])
    
    # Split by batch
    batches = adata.obs[batch_key].unique()
    mask0 = adata.obs[batch_key] == batches[0]
    mask1 = adata.obs[batch_key] == batches[1]
    b0, b1 = X[mask0], X[mask1]
    
    idx0 = np.random.choice(len(b0), min(n_sub, len(b0)), replace=False)
    idx1 = np.random.choice(len(b1), min(n_sub, len(b1)), replace=False)
    
    def rbf(A, B):
        d = np.linalg.norm(A[:, None] - B[None, :], axis=2)
        return np.exp(-d**2 / (2 * sigma**2))
    
    b0s, b1s = b0[idx0], b1[idx1]
    return rbf(b0s, b0s).mean() + rbf(b1s, b1s).mean() - 2 * rbf(b0s, b1s).mean()

## 2. Analysis

### 2.1 Retrieve Embeddings

In [299]:
# Load adata batches and concatenate for inference
adata_list = []

for subdir, _, files in os.walk(f'{st_data_silver_folder_path}/{dataset_name}'):
    for file_idx, file in enumerate(files):
        if file.endswith('.h5ad') & any(batch in file for batch in batches):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)

            print(file)

            if file == 'harmonised_merfish_mouse_brain_239-batch_batch82_shared_genes.h5ad':
                adata.obs[niche_key] = adata.obs['ccf_region_name']
                adata.obs[batch_key] = file.replace('h5ad', '')
            elif file == 'harmonised_starmap_plus_mouse_cns_batch15_shared_genes.h5ad':
                adata.obs[niche_key] = adata.obs['Sub_molecular_tissue_region']
                adata.obs[batch_key] = file.replace('h5ad', '')                

            # Filter cells without niche labels
            adata = adata[~adata.obs[niche_key].isna()]
            print(adata.obs.columns)
            adata_list.append(adata)

adata = ad.concat(adata_list, join='outer', index_unique=None)

harmonised_merfish_mouse_brain_239-batch_batch82_shared_genes.h5ad
Index(['donor_id', 'brain_section_label', 'major_brain_region',
       'ccf_region_name', 'cell_type', 'cell_id', 'n_genes',
       'nichecompass_latent_cluster', 'tissue_region', 'batch'],
      dtype='object')
harmonised_starmap_plus_mouse_cns_batch15_shared_genes.h5ad
Index(['Main_molecular_cell_type', 'Sub_molecular_cell_type',
       'Main_molecular_tissue_region', 'Sub_molecular_tissue_region',
       'Molecular_spatial_cell_type', 'dataset', 'cell_type',
       'cell_type_ontology_term_id', 'n_genes', 'cell_id',
       'nichecompass_latent_cluster', 'tissue_region', 'batch'],
      dtype='object')


/nfs/team361/sb75/.venvs/stjepa_v1/lib/python3.10/site-packages/anndata/_core/merge.py:1434: UserWarning: Only some AnnData objects have `.raw` attribute, not concatenating `.raw` attributes.
  warn(


In [ ]:
adata = harmonize_tokenize_embed_pipeline(
    adata=adata,
    sample_key=sample_key,
    batch_key=batch_key_new,
    model_folder_path=model_folder_path,
    cache_directory_path=cache_directory_path,
    gene_mapping_dict_file_path=gene_mapping_dict_file_path,
    gene_occurrence_count_file_path=gene_occurrence_count_file_path,
    gene_occurrence_count_filter_value=gene_occurrence_count_filter_value,
    ensembl_release=ensembl_release,
    species=species,
    gene_perturb_df=None,               
    nproc=nproc,
    processing_mode=processing_mode,
    harmonized_adata_save_path=f"{ablation_artifacts_folder_path}/{ablation_params['method']}/{ablation_params['model_timestamp']}/{dataset_name}.h5ad",
    save_dataset_path=f"{ablation_artifacts_folder_path}/{ablation_params['method']}/{ablation_params['model_timestamp']}/{dataset_name}.dataset",
    num_shards=num_shards,
    emb_layer=emb_layer,
    agg_excluded_genes=agg_excluded_genes,
    top_k=top_k,
    batch_size=batch_size,
    pin_memory=pin_memory,
    num_workers=num_workers,
    use_generator=use_generator,
    add_neigh_cell_ids=False,
    min_cells_per_gene=min_cells_per_gene,
    min_genes_per_cell=min_genes_per_cell)

No `sample_key` specified. Start processing entire AnnData. This is the intended behavior if the AnnData object contains only one sample. Otherwise please specify a `sample_key` or results will be invalid!!!
Harmonizing AnnData...
STEP 1: DATA VALIDATION...
Checking that adata.X contains raw counts...
✓ adata.X contains raw counts (integer values).
STEP 2: ADDING ENSEMBL IDS...
Adding ensembl IDs from release 110...
Make sure this ensembl release is aligned with pretraining.
Current ensembl release used for pretraining is 111.


INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCm39/ensembl110/Mus_musculus.GRCm39.cdna.all.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCm39/ensembl110/Mus_musculus.GRCm39.ncrna.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCm39/ensembl110/Mus_musculus.GRCm39.pep.all.fa.gz.pickle


Number of genes with matching ensembl IDs: 431.
Number of genes skipped due to non-matching ensembl IDs: 0.
STEP 3: BASIC QUALITY CONTROL...
Filtering cells with less than 10 genes.
Before cell filtering: 85704 cells.
After cell filtering: 85704 cells (removed 0 cells).
Filtering genes expressed in less than 0 cells.


INFO:stjepa.tokenizers.cell_tokenizers:Loading token dictionary from /nfs/team361/sb75/stjepa-reproducibility/artifacts/models/08042026_083724_291/token_dictionary.pkl.


Before gene filtering: 431 genes.
After gene filtering: 431 genes (removed 0 genes).
Harmonized AnnData.
Tokenizing AnnData.
STEP 1: LOADING CONFIG...
STEP 2: TOKENIZING ANNDATA OBJECT...
Filtering cells...
No 'filter_pass' column in 'adata.obs'; returning full adata.
Computing spatial neighborhood...
Normalizing gene expression counts...
Retrieving gene tokens.
Ranking gene tokens based on normalized counts (sparse version).


/nfs/team361/sb75/stjepa/src/stjepa/tokenizers/cell_tokenizers.py:678: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coding_miRNA_ids = adata.var['ensembl_id'][coding_miRNA_idx]


In [ ]:
# Write to disk
adata.write_h5ad(f"{ablation_artifacts_folder_path}/{ablation_params['method']}/{ablation_params['model_timestamp']}/{dataset_name}_embedded.h5ad")

### 2.2 Visualize UMAP

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, use_rep=emb_key,
                n_pcs=n_pcs, key_added=emb_key)
sc.tl.umap(adata, neighbors_key=emb_key)

n_pcs_string = f"_{n_pcs}_pcs" if n_pcs else ""
adata.obs[batch_key] = adata.obs[batch_key].astype('category')
categories = adata.obs[batch_key].cat.categories.tolist()
assert len(categories) == 2, "Expected exactly 2 categories for batch_key"
rename = dict(zip(sorted(categories), ["MERFISH", "STARmap PLUS"]))
adata.obs[batch_key] = adata.obs[batch_key].map(rename).astype("category")

sc.pl.umap(
    adata,
    color=batch_key,
    title="Technology",
    legend_loc="right margin",
    legend_fontsize=10,
    show=False,
)
for collection in plt.gca().collections:
    collection.set_rasterized(True)
plt.savefig(
    f"{ablation_artifacts_folder_path}/{ablation_params['method']}/{ablation_params['model_timestamp']}/{dataset_name}_{emb_key}_umap_by_{batch_key}{n_pcs_string}.svg",
    format="svg", bbox_inches="tight"
)
plt.show()
plt.close()

for label_key in label_keys:
    n_cats = adata.obs[label_key].nunique()
    palette = make_large_cmap(n_cats)
    sc.pl.umap(adata, color=label_key, palette=palette, show=False)
    for collection in plt.gca().collections:
        collection.set_rasterized(True)
    plt.savefig(
        f"{ablation_artifacts_folder_path}/{ablation_params['method']}/{ablation_params['model_timestamp']}/{dataset_name}_{emb_key}_umap_by_{label_key}{n_pcs_string}.svg",
        format="svg", bbox_inches="tight"
    )
    plt.show()
    plt.close()

In [ ]:
# Write to disk
adata.write_h5ad(f"{ablation_artifacts_folder_path}/{ablation_params['method']}/{ablation_params['model_timestamp']}/{dataset_name}_embedded.h5ad")

### 2.3 Visualize Niches

In [ ]:
adata = sc.read_h5ad(f"{ablation_artifacts_folder_path}/{ablation_params['method']}/{ablation_params['model_timestamp']}/{dataset_name}_embedded.h5ad")

In [ ]:
#ablation_params['latent_leiden_resolution'] = 0.6

In [ ]:
# Plot embedding-based clusters
latent_cluster_key = f"{emb_key}_clusters_res_{ablation_params['latent_leiden_resolution']}"

sc.tl.leiden(adata,
             neighbors_key=emb_key,
             key_added=latent_cluster_key,
             resolution=ablation_params['latent_leiden_resolution'],
             flavor="igraph",
             n_iterations=2)

In [ ]:
print(adata.obs[latent_cluster_key].nunique())
print(adata.obs[latent_cluster_key].value_counts())

In [ ]:
# Write to disk
adata.write_h5ad(f"{ablation_artifacts_folder_path}/{ablation_params['method']}/{ablation_params['model_timestamp']}/{dataset_name}_embedded_clustered.h5ad")

In [ ]:
samples = adata.obs[batch_key].unique().tolist()
for sample in samples:
    adata_sample = adata[adata.obs[batch_key] == sample]
    sc.pl.spatial(
        adata_sample,
        color=latent_cluster_key,
        palette=custom_palette,
        spot_size=spot_sizes[sample],
        title=sample,
        legend_loc="right margin",
        legend_fontsize=10,
        show=False,
    )

    fig = plt.gcf()
    for ax in fig.axes:
        for child in ax.get_children():
            if isinstance(child, mpl.image.AxesImage):
                child.set_rasterized(True)
        for collection in ax.collections:
            collection.set_rasterized(True)

    plt.savefig(
        f"{ablation_artifacts_folder_path}/{ablation_params['method']}/{ablation_params['model_timestamp']}/tissue_{sample}_by_{latent_cluster_key}.svg",
        format="svg", bbox_inches="tight"
    )
    plt.show()
    plt.close()

### 2.4 Compute Metrics

In [295]:
# Compute average cosine similarity of embeddings
X = adata.obsm[emb_key]
X = X / np.linalg.norm(X, axis=1, keepdims=True)
n = X.shape[0]
S = X.sum(axis=0)
avg_cos_sim = (S @ S - n) / (n * (n - 1))
print(f'{emb_key} average cosine similarity: {avg_cos_sim}')

cos_sim_result = pd.DataFrame([{
    "emb_key": emb_key,
    "avg_cosine_similarity": avg_cos_sim,
}])

out_path = (
    f"{ablation_artifacts_folder_path}/{ablation_params['method']}/{ablation_params['model_timestamp']}/"
    f"{dataset_name}_{emb_key}_avg_cosine_similarity.csv"
)
cos_sim_result.to_csv(out_path, index=False)
print(f"Saved → {out_path}")

neighborhood_emb average cosine similarity: 0.2653145492076874
Saved → /nfs/team361/sb75/stjepa-reproducibility/artifacts/ablation/stjepa_v2_rank_64_seq_len/08042026_083724_291/mmb0-1b_smb1-1b_1p_neighborhood_emb_avg_cosine_similarity.csv


In [296]:
# Compute NMI and ARI with region annotations
results = []
for region_key in ["Sub_molecular_tissue_region", "ccf_region_name"]:
    mask = adata.obs[region_key].notna()
    labels_true = adata.obs.loc[mask, region_key]
    labels_pred = adata.obs.loc[mask, latent_cluster_key]
    nmi = skm.normalized_mutual_info_score(labels_true, labels_pred)
    ari = skm.adjusted_rand_score(labels_true, labels_pred)
    print(f"NMI ({region_key}): {nmi:.4f}")
    print(f"ARI ({region_key}): {ari:.4f}")
    results.append({"region_key": region_key, "NMI": nmi, "ARI": ari})

df = pd.DataFrame(results)
df.to_csv(
    f"{ablation_artifacts_folder_path}/{ablation_params['method']}/{ablation_params['model_timestamp']}/{dataset_name}_{latent_cluster_key}_niche_identification_metrics.csv",
    index=False
)

NMI (Sub_molecular_tissue_region): 0.1139
ARI (Sub_molecular_tissue_region): 0.0941
NMI (ccf_region_name): 0.2705
ARI (ccf_region_name): 0.0964


In [297]:
# Compute batch integration metrics
results = []

# iLISI
batch = adata.obs[batch_key].values
knn = pynndescent(adata.obsm[emb_key], n_neighbors=90)
ilisi = float(np.mean(ilisi_knn(knn, batch)))
print(f"iLISI: {ilisi:.4f}")
results.append({"metric": "iLISI", "score": ilisi})

# ASW
asw = silhouette_score(
    adata.obsm[emb_key],
    adata.obs["batch"],
    metric="euclidean",
    sample_size=10000
)
print(f"ASW batch: {asw:.4f}")
results.append({"metric": "ASW", "score": asw})

# MMD
mmd_score = mmd_comparable(adata, emb_key, batch_key)
print(f"MMD: {mmd_score:.4f}")
results.append({"metric": "MMD", "score": mmd_score})

# Save
df = pd.DataFrame(results)
df.to_csv(
    f"{ablation_artifacts_folder_path}/{ablation_params['method']}/{ablation_params['model_timestamp']}/{dataset_name}_{emb_key}_batch_integration_metrics.csv",
    index=False
)

iLISI: 0.0000
ASW batch: 0.1057
MMD: 0.1115


In [298]:
# --- Config -----------------------------------------------------------
latent_leiden_resolution = 1.0
N_STEPS   = 4      # points above and below base resolution
STEP_SIZE = 0.1
MIN_RES   = 0.1    # Leiden resolution must be > 0

region_keys = ["Sub_molecular_tissue_region", "ccf_region_name"]

# Build resolution grid: N_STEPS below and above base, clamped to MIN_RES
# Use integer steps to avoid float accumulation errors in np.arange
base_int = round(latent_leiden_resolution / STEP_SIZE)  # e.g. 1.0/0.1 -> 10
resolutions = [
    round(step * STEP_SIZE, 1)
    for step in range(base_int - N_STEPS, base_int + N_STEPS + 1)
    if step * STEP_SIZE >= MIN_RES
]

# --- Sweep ------------------------------------------------------------
all_results = []

for res in resolutions:
    cluster_key = f"{emb_key}_clusters_res_{res}"

    # Leiden clustering
    try:
        rsc.tl.leiden(adata,
                      neighbors_key=emb_key,
                      key_added=cluster_key,
                      resolution=res,
                      flavor="igraph",
                      n_iterations=2)
    except NameError:
        sc.tl.leiden(adata,
                     neighbors_key=emb_key,
                     key_added=cluster_key,
                     resolution=res,
                     flavor="igraph",
                     n_iterations=2)

    n_clusters = adata.obs[cluster_key].nunique()
    print(f"\n── Resolution {res:.1f}  ({n_clusters} clusters) ──")

    for region_key in region_keys:
        mask        = adata.obs[region_key].notna()
        labels_true = adata.obs.loc[mask, region_key]
        labels_pred = adata.obs.loc[mask, cluster_key]

        nmi = skm.normalized_mutual_info_score(labels_true, labels_pred)
        ari = skm.adjusted_rand_score(labels_true, labels_pred)
        print(f"  {region_key:40s}  NMI={nmi:.4f}  ARI={ari:.4f}")

        all_results.append({
            "resolution": res,
            "n_clusters": n_clusters,
            "region_key": region_key,
            "NMI":        nmi,
            "ARI":        ari,
        })

# --- Save -------------------------------------------------------------
df = pd.DataFrame(all_results)

# Canonical column order
df = df[["resolution", "n_clusters", "region_key", "NMI", "ARI"]]

out_path = (
    f"{ablation_artifacts_folder_path}/{ablation_params['method']}/{ablation_params['model_timestamp']}/"
    f"{dataset_name}_{emb_key}_niche_identification_metrics_res_sweep.csv"
)
df.to_csv(out_path, index=False)
print(f"\nSaved {len(df)} rows → {out_path}")


── Resolution 0.6  (10 clusters) ──
  Sub_molecular_tissue_region               NMI=0.1211  ARI=0.0287
  ccf_region_name                           NMI=0.2308  ARI=0.0161

── Resolution 0.7  (12 clusters) ──
  Sub_molecular_tissue_region               NMI=0.1178  ARI=0.0439
  ccf_region_name                           NMI=0.2253  ARI=0.0206

── Resolution 0.8  (13 clusters) ──
  Sub_molecular_tissue_region               NMI=0.1178  ARI=0.0430
  ccf_region_name                           NMI=0.2470  ARI=0.0247

── Resolution 0.9  (16 clusters) ──
  Sub_molecular_tissue_region               NMI=0.1131  ARI=0.0640
  ccf_region_name                           NMI=0.2491  ARI=0.0456

── Resolution 1.0  (17 clusters) ──
  Sub_molecular_tissue_region               NMI=0.1125  ARI=0.0613
  ccf_region_name                           NMI=0.2579  ARI=0.0533

── Resolution 1.1  (17 clusters) ──
  Sub_molecular_tissue_region               NMI=0.1131  ARI=0.0632
  ccf_region_name                        